# Hotel Management Database — SQLite Coursework

**Original, fully synthetic data.** This notebook runs in Google Colab without installing packages. Run cells in order using **Runtime → Run all**. It creates `hotel_management.sqlite`, checks constraints and room scheduling, runs SQL analysis examples, then offers a download.

**Assessment coverage:** 10 relational tables; `bookings` has 14 columns and 1,800 rows; nominal (`booking_source`), ordinal (`service_priority`), interval (`lobby_temperature_c`), ratio (`room_total`); foreign keys and composite keys; simulated data; explanations and reproducible code.

## 1. Imports and reproducibility
A fixed random seed makes reruns reproducible. `pathlib.Path.cwd()` works in Colab, unlike `__file__` used by standalone scripts.

In [ ]:
from __future__ import annotations

import csv
import random
import sqlite3
from collections import Counter
from datetime import date, timedelta
from pathlib import Path

SEED = 20261106
RNG = random.Random(SEED)  # Repeatability is important for auditing coursework.
OUTPUT = Path.cwd() / 'hotel_management.sqlite'  # Colab-compatible: __file__ is unavailable in notebooks



## 2. Relational SQLite schema
Ten tables model hotel properties, guests, rooms, employees, bookings, services, payments and reviews. Foreign-key, `CHECK`, `UNIQUE`, and composite-primary-key constraints protect relationships and allowed values. A trigger prevents assigning a room to a booking at a different hotel.

In [ ]:
SCHEMA = '''
PRAGMA foreign_keys = ON;
CREATE TABLE hotels (
 hotel_id INTEGER PRIMARY KEY,
 hotel_name TEXT NOT NULL UNIQUE,
 city TEXT NOT NULL,
 star_rating INTEGER NOT NULL CHECK(star_rating BETWEEN 3 AND 5),
 opened_year INTEGER NOT NULL CHECK(opened_year BETWEEN 1950 AND 2026)
);
CREATE TABLE guests (
 guest_id INTEGER PRIMARY KEY,
 guest_code TEXT NOT NULL UNIQUE,
 country TEXT NOT NULL,
 loyalty_tier TEXT NOT NULL CHECK(loyalty_tier IN ('Bronze','Silver','Gold','Platinum')),
 loyalty_rank INTEGER NOT NULL CHECK(loyalty_rank BETWEEN 1 AND 4),
 registration_date TEXT NOT NULL CHECK(date(registration_date) IS NOT NULL),
 CHECK((loyalty_tier='Bronze' AND loyalty_rank=1) OR
       (loyalty_tier='Silver' AND loyalty_rank=2) OR
       (loyalty_tier='Gold' AND loyalty_rank=3) OR
       (loyalty_tier='Platinum' AND loyalty_rank=4))
);
CREATE TABLE rooms (
 room_id INTEGER PRIMARY KEY,
 hotel_id INTEGER NOT NULL REFERENCES hotels(hotel_id),
 room_number TEXT NOT NULL,
 room_type TEXT NOT NULL CHECK(room_type IN ('Standard','Deluxe','Suite','Family')),
 capacity INTEGER NOT NULL CHECK(capacity BETWEEN 1 AND 6),
 base_rate REAL NOT NULL CHECK(base_rate > 0),
 floor INTEGER NOT NULL CHECK(floor BETWEEN 1 AND 15),
 UNIQUE(hotel_id,room_number)
);
CREATE TABLE employees (
 employee_id INTEGER PRIMARY KEY,
 hotel_id INTEGER NOT NULL REFERENCES hotels(hotel_id),
 employee_code TEXT NOT NULL UNIQUE,
 job_role TEXT NOT NULL CHECK(job_role IN ('Reception','Housekeeping','Maintenance','Manager','Concierge')),
 hire_date TEXT NOT NULL CHECK(date(hire_date) IS NOT NULL),
 hourly_wage REAL NOT NULL CHECK(hourly_wage > 0)
);
CREATE TABLE bookings (
 booking_id INTEGER PRIMARY KEY,
 guest_id INTEGER NOT NULL REFERENCES guests(guest_id),
 hotel_id INTEGER NOT NULL REFERENCES hotels(hotel_id),
 check_in TEXT NOT NULL CHECK(date(check_in) IS NOT NULL),
 check_out TEXT NOT NULL CHECK(date(check_out) IS NOT NULL),
 nights INTEGER NOT NULL CHECK(nights > 0),
 guest_count INTEGER NOT NULL CHECK(guest_count > 0),
 booking_source TEXT NOT NULL CHECK(booking_source IN ('Direct','Website','Travel Agent','Corporate')),
 service_priority TEXT NOT NULL CHECK(service_priority IN ('Standard','Preferred','VIP')),
 priority_rank INTEGER NOT NULL CHECK(priority_rank BETWEEN 1 AND 3),
 lobby_temperature_c REAL CHECK(lobby_temperature_c BETWEEN 15 AND 35),
 status TEXT NOT NULL CHECK(status IN ('Completed','Confirmed','Cancelled')),
 booked_on TEXT NOT NULL CHECK(date(booked_on) IS NOT NULL),
 room_total REAL NOT NULL CHECK(room_total >= 0),
 CHECK(julianday(check_out)-julianday(check_in)=nights),
 CHECK(date(booked_on) <= date(check_in)),
 CHECK((service_priority='Standard' AND priority_rank=1) OR
       (service_priority='Preferred' AND priority_rank=2) OR
       (service_priority='VIP' AND priority_rank=3))
);
-- Composite PRIMARY KEY: each room may occur only once in a booking.
CREATE TABLE booking_rooms (
 booking_id INTEGER NOT NULL REFERENCES bookings(booking_id) ON DELETE CASCADE,
 room_id INTEGER NOT NULL REFERENCES rooms(room_id),
 nightly_rate REAL NOT NULL CHECK(nightly_rate > 0),
 PRIMARY KEY(booking_id,room_id)
);
CREATE TABLE services (
 service_id INTEGER PRIMARY KEY,
 service_name TEXT NOT NULL UNIQUE,
 unit_price REAL NOT NULL CHECK(unit_price >= 0)
);
-- Second composite key: each named service occurs at most once per booking.
CREATE TABLE booking_services (
 booking_id INTEGER NOT NULL REFERENCES bookings(booking_id) ON DELETE CASCADE,
 service_id INTEGER NOT NULL REFERENCES services(service_id),
 quantity INTEGER NOT NULL CHECK(quantity BETWEEN 1 AND 10),
 unit_price_at_purchase REAL NOT NULL CHECK(unit_price_at_purchase >= 0),
 PRIMARY KEY(booking_id,service_id)
);
CREATE TABLE payments (
 payment_id INTEGER PRIMARY KEY,
 booking_id INTEGER NOT NULL REFERENCES bookings(booking_id),
 payment_date TEXT NOT NULL CHECK(date(payment_date) IS NOT NULL),
 amount REAL NOT NULL CHECK(amount > 0),
 payment_method TEXT NOT NULL CHECK(payment_method IN ('Card','Bank Transfer','Cash')),
 payment_status TEXT NOT NULL CHECK(payment_status='Paid')
);
CREATE TABLE reviews (
 review_id INTEGER PRIMARY KEY,
 booking_id INTEGER NOT NULL UNIQUE REFERENCES bookings(booking_id),
 staff_rating INTEGER NOT NULL CHECK(staff_rating BETWEEN 1 AND 5),
 cleanliness_rating INTEGER NOT NULL CHECK(cleanliness_rating BETWEEN 1 AND 5),
 review_date TEXT NOT NULL CHECK(date(review_date) IS NOT NULL)
);
CREATE INDEX ix_booking_hotel_dates ON bookings(hotel_id,check_in,check_out);
CREATE INDEX ix_booking_guest ON bookings(guest_id);
CREATE INDEX ix_payments_booking ON payments(booking_id);
CREATE INDEX ix_rooms_hotel ON rooms(hotel_id);
-- Trigger enforces the hotel relationship even when SQL is used outside this generator.
CREATE TRIGGER validate_booking_room_hotel BEFORE INSERT ON booking_rooms
BEGIN
 SELECT RAISE(ABORT,'Room and booking must belong to the same hotel')
 WHERE (SELECT hotel_id FROM rooms WHERE room_id=NEW.room_id)
    <> (SELECT hotel_id FROM bookings WHERE booking_id=NEW.booking_id);
END;
'''




## 3. Generate synthetic entities and transactions
The generator builds randomized but controlled data. Active bookings reserve non-overlapping intervals per room; cancelled bookings do not block occupancy. Bookings may allocate several rooms; payments reconcile to the booking room total. All names and codes are fictional.

In [ ]:
def iso(d: date) -> str:
    return d.isoformat()


def create_database():
    if OUTPUT.exists():
        OUTPUT.unlink()  # Enables clean, repeatable generation, never append old rows.
    con = sqlite3.connect(OUTPUT)
    con.execute('PRAGMA foreign_keys=ON')
    con.executescript(SCHEMA)
    cities = ['Karachi','Lahore','Islamabad','Dubai','Abu Dhabi','Doha','Muscat']
    hotels = [(i, f'Aurora {city} Hotel', city, RNG.choice([3,4,4,5]), RNG.randint(1995,2022))
              for i,city in enumerate(cities,1)]
    con.executemany('INSERT INTO hotels VALUES (?,?,?,?,?)',hotels)

    tiers = [('Bronze',1),('Silver',2),('Gold',3),('Platinum',4)]
    countries = ['Pakistan','UAE','Qatar','Oman','UK','USA','Saudi Arabia','Canada']
    guests = []
    for i in range(1,701):
        tier,rank = RNG.choices(tiers,weights=[50,30,15,5])[0]
        registered = date(2022,1,1)+timedelta(days=RNG.randint(0,600))
        guests.append((i,f'GUEST-{i:05d}',RNG.choice(countries),tier,rank,iso(registered)))
    con.executemany('INSERT INTO guests VALUES (?,?,?,?,?,?)',guests)

    rooms, by_hotel = [], {i:[] for i in range(1,8)}
    for hotel in range(1,8):
        for j in range(1,41):
            room_id = (hotel-1)*40+j
            kind,cap,rate=RNG.choices([('Standard',2,95),('Deluxe',3,155),('Suite',4,270),('Family',5,225)],weights=[45,30,15,10])[0]
            rec=(room_id,hotel,f'{1+(j-1)//10}{j%10:01d}{(j-1)%10:01d}',kind,cap,round(rate*RNG.uniform(.85,1.20),2),1+(j-1)//10)
            rooms.append(rec)
            by_hotel[hotel].append(rec)
    con.executemany('INSERT INTO rooms VALUES (?,?,?,?,?,?,?)',rooms)

    roles = ['Reception','Housekeeping','Maintenance','Manager','Concierge']
    employees=[]
    for hotel in range(1,8):
        for j in range(1,16):
            i=(hotel-1)*15+j
            role = roles[(j-1)%5]
            employees.append((i,hotel,f'EMP-{i:04d}',role,iso(date(2018,1,1)+timedelta(days=RNG.randrange(2400))),round(RNG.uniform(8,32),2)))
    con.executemany('INSERT INTO employees VALUES (?,?,?,?,?,?)',employees)

    catalog=[('Airport Transfer',32),('Breakfast',18),('Spa',75),('Laundry',15),('Room Service',24),('Parking',10),('Late Checkout',35),('Guided Tour',55)]
    con.executemany('INSERT INTO services VALUES (?,?,?)',[(i,name,cost) for i,(name,cost) in enumerate(catalog,1)])

    # Each room has a list of occupied [arrival,departure) periods; cancelled bookings
    # are not kept in occupancy, and checkout/checkin on the same day is allowed.
    occupancy={r[0]:[] for r in rooms}
    booking_records=[]
    room_records=[]
    service_records=[]
    payment_records=[]
    review_records=[]
    payment_id=review_id=0
    start=date(2024,1,1)
    for booking_id in range(1,1801):
        status=RNG.choices(['Completed','Confirmed','Cancelled'],weights=[72,20,8])[0]
        # Confirmed dates are in the future relative to generator reference 2026-11-01.
        if status=='Confirmed':
            checkin=date(2026,11,2)+timedelta(days=RNG.randrange(85))
        else:
            checkin=start+timedelta(days=RNG.randrange(1035))
        nights=RNG.randint(1,7)
        checkout=checkin+timedelta(days=nights)
        hotel_id=RNG.randint(1,7)
        requested_rooms=RNG.choices([1,2,3],weights=[80,17,3])[0]
        candidates=[]
        for rec in by_hotel[hotel_id]:
            rid=rec[0]
            if status=='Cancelled' or all(checkout<=a or checkin>=b for a,b in occupancy[rid]):
                candidates.append(rec)
        if len(candidates)<requested_rooms:
            raise RuntimeError('Not enough available rooms. Adjust generator capacity.')
        selected=RNG.sample(candidates,requested_rooms)
        if status!='Cancelled':
            for rec in selected:
                occupancy[rec[0]].append((checkin,checkout))
        guest_count=RNG.randint(1,min(sum(r[4] for r in selected),requested_rooms*3))
        guest=RNG.randint(1,len(guests))
        source=RNG.choices(['Direct','Website','Travel Agent','Corporate'],weights=[20,53,17,10])[0]
        priority, rank=RNG.choices([('Standard',1),('Preferred',2),('VIP',3)],weights=[70,23,7])[0]
        # Celsius is an INTERVAL variable: differences matter, zero is arbitrary.
        lobby_temp=round(RNG.gauss(22.5,1.8),1)
        lobby_temp=max(17.0,min(29.0,lobby_temp))
        days_prior=RNG.randint(0,100)
        booked_on=max(date(2023,1,1),checkin-timedelta(days=days_prior))
        seasonal=1.20 if checkin.month in (6,7,12) else 1.0
        subtotal=0
        for rec in selected:
            price=round(rec[5]*seasonal*RNG.uniform(.92,1.10),2)
            subtotal+=price*nights
            room_records.append((booking_id,rec[0],price))
        subtotal=round(subtotal,2)
        booking_records.append((booking_id,guest,hotel_id,iso(checkin),iso(checkout),nights,guest_count,source,priority,rank,lobby_temp,status,iso(booked_on),subtotal))
        if status!='Cancelled':
            for service_id in RNG.sample(range(1,9),RNG.randint(0,3)):
                service_records.append((booking_id,service_id,RNG.randint(1,3),catalog[service_id-1][1]))
            # Each non-cancelled booking gets one or two payments, summing to room total.
            portions=[subtotal] if RNG.random()<.65 else [round(subtotal*.35,2),round(subtotal-round(subtotal*.35,2),2)]
            for value in portions:
                payment_id+=1
                payment_records.append((payment_id,booking_id,iso(booked_on),value,RNG.choice(['Card','Bank Transfer','Cash']),'Paid'))
            if status=='Completed' and RNG.random()<.62:
                review_id+=1
                review_records.append((review_id,booking_id,RNG.choices([1,2,3,4,5],[3,6,18,41,32])[0],RNG.choices([1,2,3,4,5],[2,6,16,42,34])[0],iso(checkout+timedelta(days=RNG.randint(0,10)))))

    con.executemany('INSERT INTO bookings VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)',booking_records)
    con.executemany('INSERT INTO booking_rooms VALUES (?,?,?)',room_records)
    con.executemany('INSERT INTO booking_services VALUES (?,?,?,?)',service_records)
    con.executemany('INSERT INTO payments VALUES (?,?,?,?,?,?)',payment_records)
    con.executemany('INSERT INTO reviews VALUES (?,?,?,?,?)',review_records)
    con.commit()

    # Critical validations, not merely row-count checks.
    assert con.execute('PRAGMA foreign_key_check').fetchall()==[]
    assert con.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
    assert con.execute('SELECT COUNT(*) FROM bookings').fetchone()[0]>=1000
    assert con.execute('SELECT COUNT(*) FROM booking_rooms br JOIN rooms r ON br.room_id=r.room_id JOIN bookings b ON br.booking_id=b.booking_id WHERE r.hotel_id<>b.hotel_id').fetchone()[0]==0
    assert con.execute('SELECT COUNT(*) FROM bookings b WHERE room_total <> (SELECT ROUND(SUM(br.nightly_rate*b.nights),2) FROM booking_rooms br WHERE br.booking_id=b.booking_id)').fetchone()[0]==0
    assert con.execute('''SELECT COUNT(*) FROM booking_rooms x JOIN booking_rooms y
               ON x.room_id=y.room_id AND x.booking_id<y.booking_id
               JOIN bookings a ON a.booking_id=x.booking_id
               JOIN bookings b ON b.booking_id=y.booking_id
               WHERE a.status<>'Cancelled' AND b.status<>'Cancelled'
               AND a.check_in<b.check_out AND b.check_in<a.check_out''').fetchone()[0]==0
    assert con.execute('''SELECT COUNT(*) FROM bookings b WHERE status<>'Cancelled'
        AND ABS(room_total-(SELECT ROUND(SUM(amount),2) FROM payments WHERE booking_id=b.booking_id))>0.001''').fetchone()[0]==0
    assert con.execute("SELECT COUNT(*) FROM reviews r JOIN bookings b USING(booking_id) WHERE b.status!='Completed'").fetchone()[0]==0
    print('Database created:',OUTPUT)
    for table in ['hotels','guests','rooms','employees','bookings','booking_rooms','services','booking_services','payments','reviews']:
        print(f'{table:18} {con.execute("SELECT COUNT(*) FROM "+table).fetchone()[0]:>5}')
    print('Foreign keys, room conflicts, financial totals and integrity: PASS')
    # SQL extraction examples illustrate useful analysis of relational data.
    examples={
       'occupancy_by_hotel.sql': '''SELECT h.hotel_name, COUNT(DISTINCT b.booking_id) AS active_bookings,
       ROUND(SUM(b.room_total),2) AS room_revenue
       FROM hotels h JOIN bookings b ON h.hotel_id=b.hotel_id
       WHERE b.status<>'Cancelled'
       GROUP BY h.hotel_id ORDER BY room_revenue DESC;''',
       'ratings_by_hotel.sql': '''SELECT h.hotel_name, COUNT(*) AS review_count,
       ROUND(AVG(v.staff_rating),2) AS avg_staff_rating,
       ROUND(AVG(v.cleanliness_rating),2) AS avg_cleanliness_rating
       FROM reviews v JOIN bookings b USING(booking_id)
       JOIN hotels h ON h.hotel_id=b.hotel_id
       GROUP BY h.hotel_id ORDER BY avg_staff_rating DESC;''',
       'service_pairs.sql': '''SELECT s1.service_name AS first_service, s2.service_name AS second_service,
       COUNT(*) AS co_occurrences
       FROM booking_services x JOIN booking_services y
       ON x.booking_id=y.booking_id AND x.service_id<y.service_id
       JOIN services s1 ON s1.service_id=x.service_id
       JOIN services s2 ON s2.service_id=y.service_id
       GROUP BY x.service_id,y.service_id ORDER BY co_occurrences DESC;'''
    }
    for name,query in examples.items():
        (OUTPUT.parent/name).write_text(query+'\n',encoding='utf-8')
    con.close()



## 4. Build the database and validate it
This cell **replaces** the output file if run again and prints exact table counts. It checks SQLite integrity, foreign-key integrity, room-to-hotel consistency, room occupancy overlaps, room-charge totals, payment totals and review eligibility.

In [ ]:
create_database()

## 5. Analyze the generated database using SQL
The following reads demonstrate information extraction and relationships between service purchases. The service-pair count is an association/co-occurrence example; it is not an Apriori rule or an estimate of causality.

In [ ]:
import pandas as pd
from IPython.display import display

queries = {
    "Hotel booking revenue": """
        SELECT h.hotel_name, COUNT(*) AS active_bookings, ROUND(SUM(b.room_total),2) AS room_revenue
        FROM hotels h JOIN bookings b ON h.hotel_id=b.hotel_id
        WHERE b.status <> 'Cancelled'
        GROUP BY h.hotel_id ORDER BY room_revenue DESC
    """,
    "Average staff and cleanliness ratings": """
        SELECT h.hotel_name, COUNT(*) AS review_count,
               ROUND(AVG(v.staff_rating),2) AS avg_staff_rating,
               ROUND(AVG(v.cleanliness_rating),2) AS avg_cleanliness_rating
        FROM reviews v JOIN bookings b USING(booking_id)
        JOIN hotels h ON h.hotel_id=b.hotel_id
        GROUP BY h.hotel_id ORDER BY avg_staff_rating DESC
    """,
    "Frequently purchased service pairs": """
        SELECT s1.service_name AS first_service, s2.service_name AS second_service,
               COUNT(*) AS co_occurrences
        FROM booking_services x JOIN booking_services y
             ON x.booking_id=y.booking_id AND x.service_id<y.service_id
        JOIN services s1 ON s1.service_id=x.service_id
        JOIN services s2 ON s2.service_id=y.service_id
        GROUP BY x.service_id,y.service_id ORDER BY co_occurrences DESC
        LIMIT 10
    """
}
with sqlite3.connect(OUTPUT) as connection:
    for name, query in queries.items():
        print('\n' + name)
        display(pd.read_sql_query(query, connection))

## 6. Download the SQLite database and example SQL queries
Google Colab stores notebook-generated files temporarily in its session, so download your submission artifacts. If running in local Jupyter rather than Colab, they remain in your working directory.

In [ ]:
from pathlib import Path

print('Created:', OUTPUT, '| bytes:', OUTPUT.stat().st_size)
try:
    from google.colab import files
    files.download(str(OUTPUT))
except ImportError:
    print('Not running in Colab. Database saved at:', OUTPUT)
print('SQL examples are also saved next to the SQLite database:')
for path in sorted(OUTPUT.parent.glob('*.sql')):
    if path.name in {'occupancy_by_hotel.sql', 'ratings_by_hotel.sql', 'service_pairs.sql'}:
        print(' -', path.name)